# Asclepius Synthetic Clinical Notes

Downloads `starmpcc/Asclepius-Synthetic-Clinical-Notes` from Hugging Face (~158k synthetic discharge summaries with instruction–answer pairs, CC BY-NC-SA 4.0) and saves a local copy under `data/asclepius/`.

**Part 1** downloads and saves the data (run once). **Part 2** reads the saved parquet, so you can start there without re-downloading.

# Part 1: Download (run once)

In [1]:
%pip install -q datasets pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd
from datasets import load_dataset

DATA_DIR = Path("data/asclepius")
DATA_DIR.mkdir(parents=True, exist_ok=True)

/Users/shayan/Projects/PatientInsight/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/shayan/Projects/PatientInsight/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
ds = load_dataset("starmpcc/Asclepius-Synthetic-Clinical-Notes")
ds

Generating train split: 100%|██████████| 158114/158114 [00:03<00:00, 46590.19 examples/s]


DatasetDict({
    train: Dataset({
        features: ['patient_id', 'note', 'question', 'answer', 'task'],
        num_rows: 158114
    })
})

In [4]:
df = ds["train"].to_pandas()
print(df.shape)
print(df.columns.tolist())
df.head()

(158114, 5)
['patient_id', 'note', 'question', 'answer', 'task']


,patient_id,note,question,answer,task
0,0,Discharge Summary:\n\nPatient: 60-year-old mal...,Can you provide a simplified paraphrase of the...,The healthcare team used a gradual approach to...,Paraphrasing
1,1,Discharge Summary:\n\nAdmission Date: [Insert ...,Which coreferences were resolved in the hospit...,The hospital course section resolved the coref...,Coreference Resolution
2,2,Hospital Course Summary:\n\nAdmission Date: [I...,What were the key improvements in the patient'...,"During the hospital course, the patient's medi...",Summarization
3,3,Discharge Summary:\n\nPatient: 69-year-old mal...,What roles did physical therapists have in the...,Physical therapists were responsible for ensur...,Relation Extraction
4,4,Discharge Summary:\n\nPatient Information:\n- ...,What manual airway clearance techniques were u...,The discharge summary stated that 1-2 physical...,Relation Extraction


## Quick look

In [5]:
# Task distribution (if the column exists)
if "task" in df.columns:
    display(df["task"].value_counts())

# Note length in words
if "note" in df.columns:
    display(df["note"].str.split().str.len().describe())

task
Question Answering                 20038
Abbreviation Expansion             20015
Relation Extraction                19926
Named Entity Recognition           19848
Summarization                      19756
Temporal Information Extraction    19654
Coreference Resolution             19622
Paraphrasing                       19255
Name: count, dtype: int64

count    158114.000000
mean        277.405119
std          82.476246
min          94.000000
25%         218.000000
50%         267.000000
75%         325.000000
max         869.000000
Name: note, dtype: float64

In [6]:
# One full example
row = df.sample(1, random_state=0).iloc[0]
for col, val in row.items():
    print(f"=== {col} ===\n{val}\n")

=== patient_id ===
48489

=== note ===
DISCHARGE SUMMARY:

Patient Name: [REDACTED]

Medical Record Number: [REDACTED]

Date of Admission: [REDACTED]

Date of Discharge: [REDACTED]

Admitting Diagnosis: Subcutaneous swelling on right cheek

History of Present Illness: The patient, a 29-year-old male of African descent, presented to the Plastic and Reconstructive Surgery Outpatient Department with a slow-growing, non-tender swelling on the right cheek. An excisional biopsy was performed which revealed the diagnosis of KD. The patient defaulted from the clinic after the biopsy.

Hospital Course: The patient re-presented 2 years later with a larger swelling at the same site, and a diagnosis of recurrent KD was made on histopathological examination of the re-excised tissue. The patient refused radiation therapy and was monitored closely in the outpatient department. He has shown no signs of recurrence in the past 14 months.

Discharge Diagnosis: Recurrent KD

Condition at Discharge: Stable

## Save locally

In [7]:
out = DATA_DIR / "asclepius.parquet"
df.to_parquet(out, index=False)
print(f"Saved {len(df):,} rows to {out}")

Saved 158,114 rows to data/asclepius/asclepius.parquet


# Part 2: Inspect (start here after the first run)

Reads the local parquet saved in Part 1. No download needed.

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/asclepius/asclepius.parquet")
assert DATA_PATH.exists(), f"{DATA_PATH} not found. Run Part 1 once to download and save the data."
df = pd.read_parquet(DATA_PATH)
print(df.shape)

(158114, 5)


## Inspect notes and their questions

Helpers to read full notes alongside the question/answer pairs attached to them.

In [4]:
import textwrap

# Column names (adjust here if the dataset uses different ones)
ID, NOTE, Q, A, TASK = "patient_id", "note", "question", "answer", "task"
print("Columns:", df.columns.tolist())

def wrap(text, width=110):
    return "\n".join(textwrap.fill(line, width) if line.strip() else "" for line in str(text).splitlines())

def show_note(pid, max_note_chars=None, task_filter=None):
    """Print one note followed by its question/answer pairs (optionally only those whose task contains task_filter)."""
    rows = df[df[ID] == pid]
    if task_filter:
        rows = rows[rows[TASK].str.contains(task_filter, case=False, na=False)]
    note = rows.iloc[0][NOTE]
    n_words = len(str(note).split())
    print("=" * 110)
    print(f"PATIENT {pid}  |  {n_words} words  |  {len(rows)} question(s)")
    # print("=" * 110)
    # print(wrap(note if max_note_chars is None else str(note)[:max_note_chars] + " [...]"))
    for k, (_, r) in enumerate(rows.iterrows(), 1):
        print("\n" + "-" * 110)
        print(f"Q{k} [{r.get(TASK, '?')}]: {wrap(r[Q])}")
        # print(f"\nA{k}: {wrap(r[A])}")
    print()

Columns: ['patient_id', 'note', 'question', 'answer', 'task']


In [5]:
# How many questions does each note have?
q_per_note = df.groupby(ID).size()
print(q_per_note.describe())
print("\nNotes with >1 question:", (q_per_note > 1).sum())

count    158114.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
dtype: float64

Notes with >1 question: 0


### Patients with temporal information extraction questions

Shows only notes that have a temporal-extraction question, and only those questions.

In [6]:
temporal = df[df[TASK].str.contains("temporal", case=False, na=False)]
print("Temporal task label(s):", temporal[TASK].unique().tolist())
print(f"{len(temporal):,} temporal questions across {temporal[ID].nunique():,} notes\n")

for pid in temporal[ID].drop_duplicates().sample(10, random_state=42):
    show_note(pid, task_filter="temporal")

Temporal task label(s): ['Temporal Information Extraction']
19,654 temporal questions across 19,654 notes

PATIENT 108867  |  150 words  |  1 question(s)

--------------------------------------------------------------------------------------------------------------
Q1 [Temporal Information Extraction]: What was the patient's diagnosis that required surgical intervention, and how was the procedure carried out
according to the discharge summary?

PATIENT 166988  |  186 words  |  1 question(s)

--------------------------------------------------------------------------------------------------------------
Q1 [Temporal Information Extraction]: What was the patient's CD4 cell count and plasma viral load at the time of admission?

PATIENT 85375  |  284 words  |  1 question(s)

--------------------------------------------------------------------------------------------------------------
Q1 [Temporal Information Extraction]: What ocular abnormalities were identified in the patient's eye examinat

### One example per task type

Useful for seeing what each of the task categories actually asks.

In [7]:
for task, group in df.groupby(TASK):
    print(f"\n\n######## TASK: {task}  ({len(group):,} rows) ########")
    show_note(group.sample(1, random_state=0).iloc[0][ID], max_note_chars=1500)



######## TASK: Abbreviation Expansion  (20,015 rows) ########
PATIENT 30702  |  284 words  |  1 question(s)

--------------------------------------------------------------------------------------------------------------
Q1 [Abbreviation Expansion]: What was the abbreviation expanded as "EVD" in the discharge summary related to the patient's acute
obstructive hydrocephalus?



######## TASK: Coreference Resolution  (19,622 rows) ########
PATIENT 96168  |  208 words  |  1 question(s)

--------------------------------------------------------------------------------------------------------------
Q1 [Coreference Resolution]: What coreferents are present in the given discharge summary related to the patient's treatment with Endurant
and Valiant system for AAA and TAA?



######## TASK: Named Entity Recognition  (19,848 rows) ########
PATIENT 63631  |  295 words  |  1 question(s)

--------------------------------------------------------------------------------------------------------------


# Part 3: Mine patient-independent temporal queries for agentic search

**Goal.** Build a set of temporal-information-extraction queries we can use to evaluate an agentic system that *searches* a corpus of notes and *reasons over the retrieved results*.

**Problem.** Every Asclepius question is tied to one note ("When was **the patient** discharged after **the** surgery…"). Such questions only make sense if you already know which note to read, so they're not useful as search queries over the whole corpus.

**What we want.** Questions that ask about a clinical pattern across patients and still need temporal reasoning (order of events, onset → workup → diagnosis, treatment escalation, time to discharge, and so on). For example:

| Original (note-bound) | Rewritten (corpus-level) |
|---|---|
| What were the symptoms and diagnostic tests leading to **the patient's** diagnosis of congestive heart failure? | What symptoms and diagnostic tests **typically lead to** a diagnosis of congestive heart failure? |

**Approach.**
1. Take the 19,654 `Temporal Information Extraction` questions and shuffle them.
2. Split them into batches of 50. Each batch is **one GPT-5.6 call** (about 393 calls in total).
3. In each call the model picks **at most 3** questions that are already general, or that become general with a *minimal* edit (de-personalising "the patient" into "patients", dropping note-specific details). It also returns the rewritten query. Questions that would need heavy rewriting, or that depend on note-specific facts (names, dates, a single odd case), are skipped.
4. Results are cached to a JSONL file per batch, so the run can be stopped and resumed, and you can do a small pilot before the full run.
5. Merge the results back with the source `patient_id` / note / answer, flag rewrites that changed too much, and save to parquet.

Expected output: up to about 1,180 candidate queries (3 × 393). Review a sample by hand before using them.

In [2]:
# Setup: OpenAI client and config
import json, random, re, difflib, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from tqdm.auto import tqdm

client = OpenAI(api_key=Path("openai_secret.txt").read_text().strip())

MODEL       = "gpt-5.6"     # change if your account exposes a different id (e.g. a dated snapshot)
BATCH_SIZE  = 50            # questions per LLM call
PICKS       = 3             # max questions the LLM returns per batch
SEED        = 13
MAX_WORKERS = 8             # parallel calls; lower this if you hit rate limits
OUT_DIR     = Path("data/asclepius/general_temporal_queries")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH  = OUT_DIR / f"picks_{MODEL}_bs{BATCH_SIZE}_seed{SEED}.jsonl"

/Users/shayan/Projects/PatientInsight/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Build shuffled batches of temporal questions (deterministic, so the cache stays valid across runs)
temporal = df[df[TASK] == "Temporal Information Extraction"][[ID, Q]].reset_index(drop=True)
temporal = temporal.sample(frac=1, random_state=SEED).reset_index(drop=True)

batches = [temporal.iloc[i:i + BATCH_SIZE] for i in range(0, len(temporal), BATCH_SIZE)]
print(f"{len(temporal):,} temporal questions -> {len(batches)} batches of {BATCH_SIZE}")

19,654 temporal questions -> 394 batches of 50


In [9]:
# Prompt and structured-output schema
SYSTEM_PROMPT = f"""You are helping build an evaluation set for an agentic clinical search system.
The system searches a large corpus of discharge summaries from many different patients, then reasons over
the retrieved notes to answer a question. We need questions that require TEMPORAL information extraction
(sequence of events, onset -> workup -> diagnosis, treatment changes over time, time to recovery or discharge,
what happened before or after an intervention) AND that make sense as a search over many patients,
not about one specific patient.

You will get {BATCH_SIZE} questions. Each one was written about a single patient note. Pick AT MOST {PICKS} that are
the best candidates, preferring:
  1. Questions that are ALREADY patient-independent (no edit needed), or
  2. Questions that become patient-independent with a MINIMAL edit, e.g.
     - "the patient's diagnosis of X" -> "a diagnosis of X"
     - "What were the symptoms ... leading to the patient's diagnosis of CHF?"
       -> "What symptoms and diagnostic tests typically lead to a diagnosis of congestive heart failure?"
     - drop phrases like "according to the discharge summary" or "during this hospitalization".

Rules:
  - Keep the clinical entities and the temporal or causal structure of the original. Do not invent new content.
  - The rewritten query must still need temporal reasoning over note content. Reject anything that turns into a
    plain definition or textbook question ("What is CHF?").
  - Skip questions that depend on details unique to one case (exact dates, names, one rare
    combination of events that would only match one note), or that would need heavy rewriting.
  - The rewritten query must read like a realistic clinician or researcher query over a corpus of patient notes.
  - Quality beats quantity: return fewer than {PICKS}, or none, if nothing qualifies.
  - Use the exact integer id shown for each question.
"""

RESPONSE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "temporal_query_picks",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "picks": {
                    "type": "array",
                    "maxItems": PICKS,
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "id":              {"type": "integer"},
                            "rewritten_query": {"type": "string"},
                            "edit_type":       {"type": "string", "enum": ["as_is", "minimal_edit"]},
                            "temporal_aspect": {"type": "string", "description": "Short phrase: what temporal reasoning the query needs"},
                            "quality":         {"type": "integer", "minimum": 1, "maximum": 5},
                            "rationale":       {"type": "string"},
                        },
                        "required": ["id", "rewritten_query", "edit_type", "temporal_aspect", "quality", "rationale"],
                    },
                }
            },
            "required": ["picks"],
        },
    },
}

def build_user_prompt(batch):
    lines = [f"[{i}] {q.strip()}" for i, q in enumerate(batch[Q].tolist())]
    return "Questions:\n" + "\n".join(lines)

# Preview one prompt
print(build_user_prompt(batches[0])[:1500])

Questions:
[0] What was the date of the patient's discharge from the hospital? Can you extract any details about the length of stay? Also, is there any mention of a planned follow-up and any recommended medical interventions?
[1] How long after admission was the nilotinib dosage reduced to 150 mg twice daily and what was its effect on the patient's chest symptoms?
[2] When was the corneal patch graft performed on the patient?
[3] When will the patient undergo the first TTE follow-up appointment according to the discharge instructions?
[4] When did the patient's symptoms completely recover after receiving clindamycin intravenously for 6 days and lymph node aspiration? How can temporal information extraction techniques assist in identifying this information?
[5] What was the timeline of the patient's diagnosis and treatment of non-Hodgkin's lymphoma based on the information provided in the discharge summary?
[6] How long was the patient's hospital course and what diagnostic procedure was

In [10]:
# One LLM call per batch, with retries and validation
def call_batch(batch_idx, max_retries=4):
    batch = batches[batch_idx]
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": build_user_prompt(batch)},
                ],
                response_format=RESPONSE_SCHEMA,
            )
            picks = json.loads(resp.choices[0].message.content)["picks"]
            out = []
            for p in picks[:PICKS]:
                i = p["id"]
                if not (0 <= i < len(batch)):
                    continue                      # drop hallucinated ids
                src = batch.iloc[i]
                out.append({
                    "batch_idx": batch_idx,
                    "local_id": i,
                    ID: int(src[ID]),
                    "original_question": src[Q],
                    **{k: p[k] for k in ["rewritten_query", "edit_type", "temporal_aspect", "quality", "rationale"]},
                })
            usage = resp.usage.model_dump() if resp.usage else {}
            return {"batch_idx": batch_idx, "picks": out, "usage": usage}
        except Exception as e:
            wait = 2 ** attempt * 5
            print(f"batch {batch_idx}: attempt {attempt+1} failed ({type(e).__name__}: {e}); retry in {wait}s")
            time.sleep(wait)
    return None

# Sanity check on a single batch before spending on the full run
test = call_batch(0)
for p in (test or {}).get("picks", []):
    print(f"[{p['edit_type']}, q={p['quality']}]\n  ORIG: {p['original_question']}\n  NEW:  {p['rewritten_query']}\n  WHY:  {p['rationale']}\n")

[minimal_edit, q=5]
  ORIG: What was the timeline of the patient's diagnosis and treatment of non-Hodgkin's lymphoma based on the information provided in the discharge summary?
  NEW:  What diagnosis and treatment timelines are documented for patients with non-Hodgkin's lymphoma?
  WHY:  This is a broadly applicable longitudinal query requiring reconstruction of lymphoma diagnosis and treatment events across clinical notes.

[minimal_edit, q=5]
  ORIG: What was the duration of treatment with EGFR-TKI icotinib and the time frame of subsequent disease progression, as described in the discharge summary? Can temporal information be extracted?
  NEW:  Among patients treated with EGFR-TKI icotinib, how long did treatment continue, and when did subsequent disease progression occur?
  WHY:  The query preserves the drug-specific clinical sequence and requires calculating treatment duration and locating progression afterward.

[minimal_edit, q=4]
  ORIG: When was the patient diagnosed with centr

In [11]:
# Run over all batches (resumable). Set N_BATCHES to a small number (e.g. 10) for a pilot first.
N_BATCHES = len(batches)   # e.g. 10 for a pilot

done = set()
if CACHE_PATH.exists():
    with open(CACHE_PATH) as f:
        done = {json.loads(l)["batch_idx"] for l in f if l.strip()}
todo = [b for b in range(min(N_BATCHES, len(batches))) if b not in done]
print(f"{len(done)} batches cached, {len(todo)} to run -> {CACHE_PATH}")

failed = []
with open(CACHE_PATH, "a") as f, ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(call_batch, b): b for b in todo}
    for fut in tqdm(as_completed(futures), total=len(futures)):
        res = fut.result()
        if res is None:
            failed.append(futures[fut])
            continue
        f.write(json.dumps(res) + "\n")
        f.flush()

print("Failed batches (re-run this cell to retry):", sorted(failed))

0 batches cached, 394 to run -> data/asclepius/general_temporal_queries/picks_gpt-5.6_bs50_seed13.jsonl


100%|██████████| 394/394 [06:57<00:00,  1.06s/it]

Failed batches (re-run this cell to retry): []


In [12]:
# Load picks, merge back with the note and answer, and flag large edits
records, usage = [], []
with open(CACHE_PATH) as f:
    for line in f:
        r = json.loads(line)
        records.extend(r["picks"])
        usage.append(r.get("usage", {}))

picks_df = pd.DataFrame(records)
picks_df["edit_similarity"] = [
    difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio()
    for a, b in zip(picks_df["original_question"], picks_df["rewritten_query"])
]
# Leftover patient-specific wording the rewrite should have removed
picks_df["still_patient_specific"] = picks_df["rewritten_query"].str.contains(
    r"\b(?:the patient|this patient|patient's|discharge summary|this case|her|his)\b", case=False, regex=True)

picks_df = picks_df.merge(df[[ID, NOTE, A]], on=ID, how="left")

print(f"{len(picks_df):,} picks from {picks_df['batch_idx'].nunique()} batches")
print(picks_df["edit_type"].value_counts(), "\n")
print(picks_df["quality"].value_counts().sort_index(), "\n")
print("Still patient-specific:", picks_df["still_patient_specific"].sum())
print("Low similarity (<0.6, likely over-rewritten):", (picks_df["edit_similarity"] < 0.6).sum())
tot_in  = sum(u.get("prompt_tokens", 0) for u in usage)
tot_out = sum(u.get("completion_tokens", 0) for u in usage)
print(f"Tokens: {tot_in:,} in / {tot_out:,} out")

1,181 picks from 394 batches
edit_type
minimal_edit    1145
as_is             36
Name: count, dtype: int64 

quality
1       1
2       1
3       4
4     147
5    1028
Name: count, dtype: int64 

Still patient-specific: 5
Low similarity (<0.6, likely over-rewritten): 348
Tokens: 776,354 in / 309,184 out


In [14]:
# Final filter, eyeball a sample, and save
final = picks_df[
    (picks_df["quality"] >= 4)
    & (~picks_df["still_patient_specific"])
    & (picks_df["edit_similarity"] >= 0.6)
].drop_duplicates(subset="rewritten_query").reset_index(drop=True)
print(f"{len(final):,} queries kept after filtering")

for _, r in final.sample(min(15, len(final)), random_state=0).iterrows():
    print(f"• [{r['edit_type']}, sim={r['edit_similarity']:.2f}] {r['rewritten_query']}")
    # print(f"    orig: {r['original_question']}")
    # print(f"    temporal: {r['temporal_aspect']}\n")

picks_df.to_parquet(OUT_DIR / "all_picks.parquet", index=False)
final.to_parquet(OUT_DIR / "general_temporal_queries.parquet", index=False)
print("Saved to", OUT_DIR)

826 queries kept after filtering
• [minimal_edit, sim=0.66] Among patients diagnosed with Serratia marcescens pneumonia, what timeline of events led to the diagnosis, and what treatments were administered afterward during the hospital course?
• [minimal_edit, sim=0.72] What temporal relationships are documented between chemotherapy cycles and the onset and resolution of neurological deficits?
• [minimal_edit, sim=0.68] What was the time interval between a diagnosis of primary biliary cholangitis (PBC) and initiation of ursodeoxycholic acid, and what subsequent changes occurred in liver function tests?
• [minimal_edit, sim=0.70] What treatment regimens were used after a second relapse of AML, and what were the outcomes of salvage therapy?
• [as_is, sim=0.85] When was PCI of the SVG to RCA performed, and how many days after the initial coronary angiogram did it occur?
• [minimal_edit, sim=0.91] When were patients diagnosed with calciphylaxis, and how long were they treated with sodium th

In [15]:
# Final queries file: one JSON object per line -> {"query", "patient_id", "answer"}
# Note: "answer" is the gold answer to the ORIGINAL (note-bound) question, kept as a reference.
FINAL_PATH = OUT_DIR / "temporal_information_queries.jsonl"

with open(FINAL_PATH, "w") as f:
    for _, r in final.iterrows():
        f.write(json.dumps({
            "query": r["rewritten_query"],
            "patient_id": int(r[ID]),
            "answer": r[A],
        }, ensure_ascii=False) + "\n")

print(f"Wrote {len(final):,} queries to {FINAL_PATH}")
with open(FINAL_PATH) as f:
    print(f.readline()[:500])

Wrote 826 queries to data/asclepius/general_temporal_queries/temporal_information_queries.jsonl
{"query": "What timelines are documented for patients diagnosed with precocious puberty, including milestones of physical development, laboratory testing, and interventions?", "patient_id": 12634, "answer": "The patient's precocious puberty was initially confirmed with an advanced bone age x-ray and mid-afternoon laboratory tests at 9 years 7 months old. At 11 years 0 months, the histrelin implant was removed as per parental concern of menarche. At 11 years 10 months, the patient showed TS IV br
